# Producto U1 — Nick Saim Mayta Jara

**Dimensión:** Volumen de tráfico por flujo (`bytes_per_s`) — U1 batch, regresión

**Rol en el equipo:** Arquitectura Lambda y observabilidad general, coordinación técnica

**Curso:** Big Data · lambda26 · Proyecto Sello (equipo LLSW3, sección GU)

**Caso:** análisis de tráfico de red del campus universitario (dataset propio, ~400 000 flujos, capturado vía Suricata)

**Metodología:** CRISP-DM — Fases 1 a 5 (hasta modelado/evaluación; el despliegue es Unidad 2)


## Arquitectura Big Data (contexto — no es una fase de CRISP-DM)

Este notebook implementa la **ruta batch** de la arquitectura **Lambda** declarada en el
[Brief técnico-analítico](../../docs/proyecto-sello/brief.md) (Hito S2): capa batch (este
notebook) + capa de velocidad (Kafka, contenido de Unidad 2). El detalle completo de la
decisión Lambda vs. Kappa está en el brief.

## Fase 1 — Comprensión del negocio (CRISP-DM)

**Pregunta de negocio (dimensión propia):** ¿Cuál ha sido el volumen histórico de tráfico
(bytes/segundo) de los flujos capturados, y qué volumen se puede esperar según sus
características?

**Objetivo de minería de datos:** entrenar un modelo de **regresión** que prediga
`bytes_per_s` a partir de las demás variables del flujo.

**Decisión que habilita:** anticipar picos de carga y priorizar capacidad de red antes de
que ocurra congestión.

**Criterio de éxito:** el modelo debe superar con margen claro una línea base ingenua
(predecir siempre el promedio histórico de `bytes_per_s`) en RMSE, con un R² que dé
confianza para planificación (referencia orientativa: R² > 0.5 — ajustar una vez calculada
la línea base real en la Fase 2).


## Fase 2 — Comprensión de los datos (CRISP-DM)

### Extracción con esquema explícito


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType, DoubleType
)
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("u1-producto-nick")
    .getOrCreate()
)

RUTA_DATOS = "/opt/data/TRCU.csv"

# Hallazgo documentado en S03: con header=True + StructType explícito, Spark asigna
# los campos por POSICION, no por nombre de columna. Si el orden de StructField no
# coincide exactamente con el orden físico del CSV, los valores se corrompen en
# silencio (sin lanzar error). Por eso el orden de abajo respeta el header real
# observado en el dataset, columna por columna.

esquema_flujos = StructType([
    StructField("flow_id", StringType(), True),
    StructField("src_addr", StringType(), True),
    StructField("src_port", IntegerType(), True),
    StructField("dst_addr", StringType(), True),
    StructField("dst_port", IntegerType(), True),
    StructField("ip_prot", IntegerType(), True),
    StructField("timestamp", LongType(), True),
    StructField("flow_duration", DoubleType(), True),
    StructField("down_up_ratio", DoubleType(), True),
    StructField("pkt_len_max", DoubleType(), True),
    StructField("pkt_len_min", DoubleType(), True),
    StructField("pkt_len_mean", DoubleType(), True),
    StructField("pkt_len_var", DoubleType(), True),
    StructField("pkt_len_std", DoubleType(), True),
    StructField("bytes_per_s", DoubleType(), True),
    StructField("pkt_per_s", DoubleType(), True),
    StructField("fwd_pkt_per_s", DoubleType(), True),
    StructField("bwd_pkt_per_s", DoubleType(), True),
    StructField("fwd_pkt_cnt", IntegerType(), True),
    StructField("fwd_pkt_len_tot", DoubleType(), True),
    StructField("fwd_pkt_len_max", DoubleType(), True),
    StructField("fwd_pkt_len_min", DoubleType(), True),
    StructField("fwd_pkt_len_mean", DoubleType(), True),
    StructField("fwd_pkt_len_std", DoubleType(), True),
    StructField("fwd_pkt_hdr_len_tot", IntegerType(), True),
    StructField("fwd_pkt_hdr_len_min", IntegerType(), True),
    StructField("fwd_non_empty_pkt_cnt", IntegerType(), True),
    StructField("bwd_pkt_cnt", IntegerType(), True),
    StructField("bwd_pkt_len_tot", DoubleType(), True),
    StructField("bwd_pkt_len_max", DoubleType(), True),
    StructField("bwd_pkt_len_min", DoubleType(), True),
    StructField("bwd_pkt_len_mean", DoubleType(), True),
    StructField("bwd_pkt_len_std", DoubleType(), True),
    StructField("bwd_pkt_hdr_len_tot", IntegerType(), True),
    StructField("bwd_pkt_hdr_len_min", IntegerType(), True),
    StructField("bwd_non_empty_pkt_cnt", IntegerType(), True),
    StructField("iat_max", DoubleType(), True),
    StructField("iat_min", DoubleType(), True),
    StructField("iat_mean", DoubleType(), True),
    StructField("iat_std", DoubleType(), True),
    StructField("fwd_iat_tot", DoubleType(), True),
    StructField("fwd_iat_max", DoubleType(), True),
    StructField("fwd_iat_min", DoubleType(), True),
    StructField("fwd_iat_mean", DoubleType(), True),
    StructField("fwd_iat_std", DoubleType(), True),
    StructField("bwd_iat_tot", DoubleType(), True),
    StructField("bwd_iat_max", DoubleType(), True),
    StructField("bwd_iat_min", DoubleType(), True),
    StructField("bwd_iat_mean", DoubleType(), True),
    StructField("bwd_iat_std", DoubleType(), True),
    StructField("active_max", DoubleType(), True),
    StructField("active_min", DoubleType(), True),
    StructField("active_mean", DoubleType(), True),
    StructField("active_std", DoubleType(), True),
    StructField("idle_max", DoubleType(), True),
    StructField("idle_min", DoubleType(), True),
    StructField("idle_mean", DoubleType(), True),
    StructField("idle_std", DoubleType(), True),
    StructField("flag_SYN", IntegerType(), True),
    StructField("flag_fin", IntegerType(), True),
    StructField("flag_rst", IntegerType(), True),
    StructField("flag_ack", IntegerType(), True),
    StructField("flag_psh", IntegerType(), True),
    StructField("fwd_flag_psh", IntegerType(), True),
    StructField("bwd_flag_psh", IntegerType(), True),
    StructField("flag_urg", IntegerType(), True),
    StructField("fwd_flag_urg", IntegerType(), True),
    StructField("bwd_flag_urg", IntegerType(), True),
    StructField("flag_cwr", IntegerType(), True),
    StructField("flag_ece", IntegerType(), True),
    StructField("fwd_bulk_bytes_mean", DoubleType(), True),
    StructField("fwd_bulk_pkt_mean", DoubleType(), True),
    StructField("fwd_bulk_rate_mean", DoubleType(), True),
    StructField("bwd_bulk_bytes_mean", DoubleType(), True),
    StructField("bwd_bulk_pkt_mean", DoubleType(), True),
    StructField("bwd_bulk_rate_mean", DoubleType(), True),
    StructField("fwd_subflow_bytes_mean", DoubleType(), True),
    StructField("fwd_subflow_pkt_mean", DoubleType(), True),
    StructField("bwd_subflow_bytes_mean", DoubleType(), True),
    StructField("bwd_subflow_pkt_mean", DoubleType(), True),
    StructField("fwd_tcp_init_win_bytes", IntegerType(), True),
    StructField("bwd_tcp_init_win_bytes", IntegerType(), True),
    StructField("label", StringType(), True),
])

df = spark.read.csv(RUTA_DATOS, header=True, schema=esquema_flujos)

with open(RUTA_DATOS, "r", encoding="utf-8") as f:
    cabecera_real = f.readline().strip().split(",")
assert cabecera_real == [c.name for c in esquema_flujos.fields], (
    "El orden del esquema no coincide con el header real del CSV — revisar antes de continuar."
)

df.printSchema()
df.show(5, truncate=False)
print("Filas totales:", df.count())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/11 02:33:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- flow_id: string (nullable = true)
 |-- src_addr: string (nullable = true)
 |-- src_port: integer (nullable = true)
 |-- dst_addr: string (nullable = true)
 |-- dst_port: integer (nullable = true)
 |-- ip_prot: integer (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- flow_duration: double (nullable = true)
 |-- down_up_ratio: double (nullable = true)
 |-- pkt_len_max: double (nullable = true)
 |-- pkt_len_min: double (nullable = true)
 |-- pkt_len_mean: double (nullable = true)
 |-- pkt_len_var: double (nullable = true)
 |-- pkt_len_std: double (nullable = true)
 |-- bytes_per_s: double (nullable = true)
 |-- pkt_per_s: double (nullable = true)
 |-- fwd_pkt_per_s: double (nullable = true)
 |-- bwd_pkt_per_s: double (nullable = true)
 |-- fwd_pkt_cnt: integer (nullable = true)
 |-- fwd_pkt_len_tot: double (nullable = true)
 |-- fwd_pkt_len_max: double (nullable = true)
 |-- fwd_pkt_len_min: double (nullable = true)
 |-- fwd_pkt_len_mean: double (nullable = true)
 |

26/09/11 02:33:10 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------------------------------------+--------------+--------+--------------+--------+-------+----------------+-------------+-------------+-----------+-----------+------------+-------------+-----------+------------+---------+-------------+-------------+-----------+---------------+---------------+---------------+----------------+---------------+-------------------+-------------------+---------------------+-----------+---------------+---------------+---------------+----------------+---------------+-------------------+-------------------+---------------------+--------+-------+------------+-------------+-----------+-----------+-----------+------------+-------------+-----------+-----------+-----------+------------+-------------+----------+----------+-----------+----------+--------+--------+---------+--------+--------+--------+--------+--------+--------+------------+------------+--------+------------+------------+--------+--------+-------------------+-----------------+------------------

Filas totales: 397354


### Exploración inicial (EDA)


In [2]:
print("Resumen estadistico de variables clave:")
df.select("bytes_per_s", "flow_duration", "pkt_len_mean", "iat_mean").describe().show()

print("Nulos por columna clave:")
for columna in ["bytes_per_s", "flow_duration", "pkt_len_mean", "ip_prot"]:
    n_nulos = df.filter(F.col(columna).isNull()).count()
    print(f"  {columna}: {n_nulos} nulos")

percentiles = df.approxQuantile("bytes_per_s", [0.0, 0.25, 0.5, 0.75, 1.0], 0.01)
print("bytes_per_s -> min/p25/mediana/p75/max:", percentiles)

linea_base_ingenua = df.select(F.avg("bytes_per_s")).first()[0]
print("Linea base ingenua (promedio historico de bytes_per_s):", linea_base_ingenua)

print("Distribucion por protocolo (ip_prot):")
df.groupBy("ip_prot").count().orderBy(F.desc("count")).show()


Resumen estadistico de variables clave:


+-------+-------------------+--------------------+------------------+--------------------+
|summary|        bytes_per_s|       flow_duration|      pkt_len_mean|            iat_mean|
+-------+-------------------+--------------------+------------------+--------------------+
|  count|             397354|              397354|            397354|              397354|
|   mean|  550136.9126835192|1.0620946267620308E7|192.97075318445522|   3963328.166860908|
| stddev|2.022651824476482E7| 3.026542849501702E7|166.78199958878616|1.3829478112778643E7|
|    min|                0.0|                 0.0|               0.0|                 0.0|
|    max|             2.76E9|        1.19999999E8|            1472.0|        1.19999996E8|
+-------+-------------------+--------------------+------------------+--------------------+

Nulos por columna clave:


  bytes_per_s: 0 nulos


  flow_duration: 0 nulos


  pkt_len_mean: 0 nulos


  ip_prot: 0 nulos


bytes_per_s -> min/p25/mediana/p75/max: [0.0, 0.0, 0.0, 0.0, 2760000000.0]


Linea base ingenua (promedio historico de bytes_per_s): 550136.9126835192
Distribucion por protocolo (ip_prot):


+-------+------+
|ip_prot| count|
+-------+------+
|     17|374483|
|      6| 22579|
|      2|   243|
|      1|    49|
+-------+------+



## Fase 3 — Preparación de los datos (CRISP-DM)

### Transformación y agregación


In [3]:
df_nick = (
    df
    .withColumn(
        "protocolo",
        F.when(F.col("ip_prot") == 6, F.lit("TCP"))
         .when(F.col("ip_prot") == 17, F.lit("UDP"))
         .otherwise(F.lit("OTRO"))
    )
    .filter(F.col("bytes_per_s").isNotNull())
)

df_nick.explain(True)  # confirmar en que punto Spark deja de ser perezoso

resumen_volumen = (
    df_nick.groupBy("protocolo")
    .agg(
        F.avg("bytes_per_s").alias("bytes_per_s_prom"),
        F.max("bytes_per_s").alias("bytes_per_s_max"),
        F.count("*").alias("n_flujos"),
    )
)
resumen_volumen.show()


== Parsed Logical Plan ==
'Filter 'isNotNull('bytes_per_s)
+- Project [flow_id#0, src_addr#1, src_port#2, dst_addr#3, dst_port#4, ip_prot#5, timestamp#6L, flow_duration#7, down_up_ratio#8, pkt_len_max#9, pkt_len_min#10, pkt_len_mean#11, pkt_len_var#12, pkt_len_std#13, bytes_per_s#14, pkt_per_s#15, fwd_pkt_per_s#16, bwd_pkt_per_s#17, fwd_pkt_cnt#18, fwd_pkt_len_tot#19, fwd_pkt_len_max#20, fwd_pkt_len_min#21, fwd_pkt_len_mean#22, fwd_pkt_len_std#23, fwd_pkt_hdr_len_tot#24, ... 59 more fields]
   +- Relation [flow_id#0,src_addr#1,src_port#2,dst_addr#3,dst_port#4,ip_prot#5,timestamp#6L,flow_duration#7,down_up_ratio#8,pkt_len_max#9,pkt_len_min#10,pkt_len_mean#11,pkt_len_var#12,pkt_len_std#13,bytes_per_s#14,pkt_per_s#15,fwd_pkt_per_s#16,bwd_pkt_per_s#17,fwd_pkt_cnt#18,fwd_pkt_len_tot#19,fwd_pkt_len_max#20,fwd_pkt_len_min#21,fwd_pkt_len_mean#22,fwd_pkt_len_std#23,fwd_pkt_hdr_len_tot#24,... 58 more fields] csv

== Analyzed Logical Plan ==
flow_id: string, src_addr: string, src_port: int, dst_a

+---------+------------------+---------------+--------+
|protocolo|  bytes_per_s_prom|bytes_per_s_max|n_flujos|
+---------+------------------+---------------+--------+
|     OTRO|  8538.14210946233| 1290322.580645|     292|
|      UDP|209179.59557949615|          2.4E9|  374483|
|      TCP| 6212073.483039771|         2.76E9|   22579|
+---------+------------------+---------------+--------+



### Calidad de datos y particionamiento analítico


In [4]:
df_dedup = df_nick.dropDuplicates(["flow_id"])

df_limpio = df_dedup.na.fill({
    "bytes_per_s": 0.0,
    "pkt_len_mean": 0.0,
    "iat_mean": 0.0,
    "active_mean": 0.0,
    "idle_mean": 0.0,
})

RUTA_SALIDA = "/opt/artifacts/nick/flujos_particionado"
(
    df_limpio.write.mode("overwrite")
    .partitionBy("protocolo")
    .parquet(RUTA_SALIDA)
)

df_verificacion = spark.read.parquet(RUTA_SALIDA)
print("Filas tras deduplicacion:", df_limpio.count())
print("Filas leidas de vuelta desde Parquet:", df_verificacion.count())

df_verificacion.filter(F.col("protocolo") == "TCP").explain(True)  # confirmar PartitionFilters

26/09/11 02:33:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/09/11 02:33:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/09/11 02:33:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/09/11 02:33:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/09/11 02:33:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/09/11 02:33:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/09/11 02:33:25 WARN MemoryManager: Total allocation exceeds 95.

26/09/11 02:33:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 40.00% for 19 writers
26/09/11 02:33:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 42.22% for 18 writers
26/09/11 02:33:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 44.71% for 17 writers
26/09/11 02:33:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 47.50% for 16 writers
26/09/11 02:33:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 50.67% for 15 writers
26/09/11 02:33:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 54.29% for 14 writers
26/09/11 02:33:25 WARN MemoryManager: Total allocation exceeds 9

26/09/11 02:33:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/09/11 02:33:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/09/11 02:33:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/09/11 02:33:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/09/11 02:33:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/09/11 02:33:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/09/11 02:33:26 WARN MemoryManager: Total allocation exceeds 95.

26/09/11 02:33:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/09/11 02:33:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/09/11 02:33:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/09/11 02:33:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/09/11 02:33:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/09/11 02:33:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/09/11 02:33:26 WARN MemoryManager: Total allocation exceeds 95.

26/09/11 02:33:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 54.29% for 14 writers
26/09/11 02:33:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/09/11 02:33:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/09/11 02:33:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/09/11 02:33:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/09/11 02:33:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/09/11 02:33:26 WARN MemoryManager: Total allocation exceeds 95

Filas tras deduplicacion: 133311


Filas leidas de vuelta desde Parquet: 133311
== Parsed Logical Plan ==
'Filter '`=`('protocolo, TCP)
+- Relation [flow_id#2259,src_addr#2260,src_port#2261,dst_addr#2262,dst_port#2263,ip_prot#2264,timestamp#2265L,flow_duration#2266,down_up_ratio#2267,pkt_len_max#2268,pkt_len_min#2269,pkt_len_mean#2270,pkt_len_var#2271,pkt_len_std#2272,bytes_per_s#2273,pkt_per_s#2274,fwd_pkt_per_s#2275,bwd_pkt_per_s#2276,fwd_pkt_cnt#2277,fwd_pkt_len_tot#2278,fwd_pkt_len_max#2279,fwd_pkt_len_min#2280,fwd_pkt_len_mean#2281,fwd_pkt_len_std#2282,fwd_pkt_hdr_len_tot#2283,... 59 more fields] parquet

== Analyzed Logical Plan ==
flow_id: string, src_addr: string, src_port: int, dst_addr: string, dst_port: int, ip_prot: int, timestamp: bigint, flow_duration: double, down_up_ratio: double, pkt_len_max: double, pkt_len_min: double, pkt_len_mean: double, pkt_len_var: double, pkt_len_std: double, bytes_per_s: double, pkt_per_s: double, fwd_pkt_per_s: double, bwd_pkt_per_s: double, fwd_pkt_cnt: int, fwd_pkt_len_tot: 

### Selección de predictores y ensamblado del vector de features


In [5]:
from pyspark.ml.feature import VectorAssembler

predictores_nick = [
    "ip_prot", "flow_duration", "pkt_len_mean", "pkt_len_std",
    "fwd_pkt_len_mean", "bwd_pkt_len_mean", "iat_mean",
    "active_mean", "idle_mean", "flag_SYN", "flag_ack",
    "fwd_tcp_init_win_bytes", "bwd_tcp_init_win_bytes",
]

ensamblador = VectorAssembler(inputCols=predictores_nick, outputCol="features", handleInvalid="skip")
dataset_ml = (
    ensamblador.transform(df_limpio)
    .select("features", F.col("bytes_per_s").alias("valor_real"))
)

df_train, df_test = dataset_ml.randomSplit([0.8, 0.2], seed=42)
print("Filas de entrenamiento:", df_train.count(), " / Filas de prueba:", df_test.count())


Filas de entrenamiento: 106874  / Filas de prueba: 26437


## Fase 4 — Modelado (CRISP-DM)


In [6]:
from pyspark.ml.regression import LinearRegression, RandomForestRegressor

configuraciones_nick = {
    "LinearRegression base": LinearRegression(featuresCol="features", labelCol="valor_real"),
    "LinearRegression + Ridge (L2)": LinearRegression(featuresCol="features", labelCol="valor_real", regParam=0.1, elasticNetParam=0.0),
    "LinearRegression + Lasso (L1)": LinearRegression(featuresCol="features", labelCol="valor_real", regParam=0.1, elasticNetParam=1.0),
    "LinearRegression + Elastic Net": LinearRegression(featuresCol="features", labelCol="valor_real", regParam=0.1, elasticNetParam=0.5),
    "RandomForestRegressor": RandomForestRegressor(featuresCol="features", labelCol="valor_real", seed=42),
}

modelos_entrenados_nick = {}
predicciones_nick = {}
for nombre, estimador in configuraciones_nick.items():
    modelo = estimador.fit(df_train)
    modelos_entrenados_nick[nombre] = modelo
    predicciones_nick[nombre] = modelo.transform(df_test)
    print("Entrenado:", nombre)


26/09/11 02:33:43 WARN Instrumentation: [1ed4c604] regParam is zero, which might cause numerical instability and overfitting.


netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory


netlib-lapack: JNI_OnLoad: dlopen(liblapack.so.3) failed: liblapack.so.3: cannot open shared object file: No such file or directory


Entrenado: LinearRegression base


Entrenado: LinearRegression + Ridge (L2)


Entrenado: LinearRegression + Lasso (L1)


Entrenado: LinearRegression + Elastic Net


Entrenado: RandomForestRegressor


## Fase 5 — Evaluación (CRISP-DM)


In [7]:
from pyspark.ml.evaluation import RegressionEvaluator

ev_rmse = RegressionEvaluator(labelCol="valor_real", metricName="rmse")
ev_r2 = RegressionEvaluator(labelCol="valor_real", metricName="r2")
ev_mae = RegressionEvaluator(labelCol="valor_real", metricName="mae")

pred_baseline = df_test.withColumn("prediction", F.lit(linea_base_ingenua))
rmse_base = ev_rmse.evaluate(pred_baseline)
print(f"Linea base ingenua -> RMSE={rmse_base:.4f}\n")

resultados_nick = {}
for nombre, pred in predicciones_nick.items():
    rmse = ev_rmse.evaluate(pred)
    r2 = ev_r2.evaluate(pred)
    mae = ev_mae.evaluate(pred)
    supera_base = "SI" if rmse < rmse_base else "NO"
    resultados_nick[nombre] = rmse
    print(f"{nombre:32s} RMSE={rmse:.4f}  R2={r2:.4f}  MAE={mae:.4f}  (supera linea base: {supera_base})")

nombre_ganador = min(resultados_nick, key=resultados_nick.get)
print(f"\nModelo ganador (menor RMSE): {nombre_ganador}")

modelo_ganador = modelos_entrenados_nick[nombre_ganador]
modelo_ganador.write().overwrite().save("/opt/artifacts/nick/modelo_volumen")
print("Modelo ganador guardado en /opt/artifacts/nick/modelo_volumen")

Linea base ingenua -> RMSE=34245209.1615



LinearRegression base            RMSE=33787402.3774  R2=0.0259  MAE=3159715.7261  (supera linea base: SI)


LinearRegression + Ridge (L2)    RMSE=33787402.3733  R2=0.0259  MAE=3159715.7349  (supera linea base: SI)


LinearRegression + Lasso (L1)    RMSE=33786414.8927  R2=0.0260  MAE=3164290.7875  (supera linea base: SI)


LinearRegression + Elastic Net   RMSE=33786414.8964  R2=0.0260  MAE=3164290.7783  (supera linea base: SI)


RandomForestRegressor            RMSE=16116512.4038  R2=0.7784  MAE=638386.1837  (supera linea base: SI)

Modelo ganador (menor RMSE): RandomForestRegressor


Modelo ganador guardado en /opt/artifacts/nick/modelo_volumen


## Cierre de fases y alcance

Este notebook cubre las **5 fases de CRISP-DM hasta el modelado/evaluación**: Comprensión del
negocio → Comprensión de los datos → Preparación de los datos → Modelado → Evaluación.
**No incluye la Fase 6 (Despliegue):** poner el modelo a inferir sobre flujos en vivo es
contenido de Unidad 2 (Spark Structured Streaming + Kafka), declarado como dimensión U2 en
el brief.

## Hallazgo(s) de esta dimensión

Ejecutado de punta a punta contra `TRCU.csv` (397 354 flujos reales capturados por Suricata):

- **Calidad de datos:** la deduplicación por `flow_id` eliminó el 66.5% de los registros
  (397 354 → 133 311 filas), es decir, casi dos tercios de las capturas llegan repetidas al
  pipeline batch — un hallazgo de calidad de la fuente Suricata, no del tráfico en sí, y una
  señal a revisar con el equipo antes de escalar a la ingesta en vivo (Unidad 2).
- **Modelado:** `bytes_per_s` tiene una relación fuertemente no lineal con las features del
  flujo — los 4 modelos lineales (base, Ridge, Lasso, Elastic Net) apenas superan la línea
  base ingenua (R² ≈ 0.026 en los cuatro), mientras que `RandomForestRegressor` alcanza
  **R² = 0.7784** y reduce el RMSE en ~53% frente a la línea base (16.1M vs. 34.2M),
  cumpliendo el criterio de éxito de la Fase 1 (R² > 0.5). Es el modelo guardado como ganador.
- **Distribución del tráfico:** UDP domina en número de flujos (374 483 de 397 354, ~94%),
  pero TCP tiene el `bytes_per_s` promedio más alto (6.21M vs. 209K de UDP) — el volumen de
  tráfico por flujo no se explica solo por el protocolo dominante en cantidad de conexiones.

## Cómo ejecutar este notebook (evidencia de contribución)

1. Levantar el laboratorio (`docker compose up -d` desde `pyspark/`).
2. El dataset real (`TRCU.csv`) ya está en `pyspark/data/`, montado en `/opt/data/` dentro del contenedor — no requiere ajustar `RUTA_DATOS`.
3. Ejecutar de punta a punta (`Run All`), sin intervención manual: el modelo ganador se elige y se guarda automáticamente en la Fase 5.
4. Confirmar la carpeta de salida (`!ls -R` o `os.walk`) sobre `/opt/artifacts/nick/` (Parquet particionado + modelo guardado).
5. Capturar pantalla con reloj del sistema y usuario/perfil visibles.
6. Commit del notebook ejecutado al repositorio del equipo (`pyspark/artifacts/` no se versiona, ver `.gitignore`).